# Deep Dive: Cracking All 6 Puzzle Types + SFT Training Data Prep

**NVIDIA Nemotron Model Reasoning Challenge**

In this notebook, we'll perform a comprehensive exploration of the competition dataset, a collection of **logical reasoning puzzles** where a hidden transformation rule must be discovered from examples and applied to a new input.

### What we'll cover:
1. **Data Overview**: structure, sizes, formats
2. **Problem Type Classification**: regex-based classifier for all 6 puzzle categories
3. **Deep Dive per Puzzle Type**: sample prompts, answer formats, key patterns
4. **Pattern Mining**: reverse-engineering the hidden rules
5. **Answer Format Analysis**: critical for `\boxed{}` output formatting
6. **SFT Training Data Preparation**: chat-formatted JSONL ready for LoRA fine-tuning
7. **Strategy Recommendations**: what to focus on for maximum leaderboard gain

> **Competition constraint:** You can ONLY submit a LoRA adapter (rank ≤ 32). No prompt changes, no external scripts at inference time.

In [ ]:
import json
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Style settings
sns.set_theme(style="whitegrid", palette="husl")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 12

# Color palette for the 6 puzzle types
PUZZLE_COLORS = {
    "Number Base Conversion": "#FF6B6B",
    "Gravitational Constant": "#4ECDC4",
    "Equation Transformation": "#45B7D1",
    "Text Encryption": "#96CEB4",
    "Bit Manipulation": "#FFEAA7",
    "Unit Conversion": "#DDA0DD",
}

print("Libraries loaded successfully.")

## 1. Load & Preview the Data

In [ ]:
# Load the competition data
DATA_DIR = "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge"
train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
test_df = pd.read_csv(f"{DATA_DIR}/test.csv")

print(f"Training set: {train_df.shape[0]} rows, {train_df.shape[1]} columns")
print(f"Test set:     {test_df.shape[0]} rows, {test_df.shape[1]} columns")
print(f"\nColumns: {list(train_df.columns)}")
print("\n--- Training Data Info ---")
train_df.info()
print("\n--- First 3 rows ---")
train_df.head(3)

In [ ]:
# Let's look at a full prompt to understand the structure
print("=" * 80)
print("SAMPLE PROMPT (first row)")
print("=" * 80)
print(train_df["prompt"].iloc[0])
print("=" * 80)
print(f"\nANSWER: {train_df['answer'].iloc[0]}")

In [ ]:
# Basic statistics on prompt and answer lengths
train_df["prompt_len"] = train_df["prompt"].str.len()
train_df["prompt_words"] = train_df["prompt"].str.split().str.len()
train_df["answer_len"] = train_df["answer"].astype(str).str.len()
train_df["est_tokens"] = train_df["prompt_words"] * 1.3  # rough token estimate

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(train_df["prompt_len"], bins=30, color="#45B7D1", edgecolor="white", alpha=0.8)
axes[0].set_title("Prompt Length (characters)", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Characters")
axes[0].set_ylabel("Count")

axes[1].hist(train_df["prompt_words"], bins=30, color="#FF6B6B", edgecolor="white", alpha=0.8)
axes[1].set_title("Prompt Length (words)", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Words")

axes[2].hist(train_df["answer_len"], bins=30, color="#96CEB4", edgecolor="white", alpha=0.8)
axes[2].set_title("Answer Length (characters)", fontsize=14, fontweight="bold")
axes[2].set_xlabel("Characters")

plt.suptitle("Distribution of Prompt and Answer Lengths", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print(
    f"Prompt length (chars): min={train_df['prompt_len'].min()}, "
    f"median={train_df['prompt_len'].median():.0f}, max={train_df['prompt_len'].max()}"
)
print(
    f"Estimated tokens:      min={train_df['est_tokens'].min():.0f}, "
    f"median={train_df['est_tokens'].median():.0f}, max={train_df['est_tokens'].max():.0f}"
)
print(
    f"Answer length (chars): min={train_df['answer_len'].min()}, "
    f"median={train_df['answer_len'].median():.0f}, max={train_df['answer_len'].max()}"
)

## 2. Problem Type Classification

The benchmark contains **6 distinct puzzle types**. Each prompt describes a "secret world" where a hidden rule transforms inputs. We need to figure out the rule from examples and apply it.

Let's classify every row using keyword matching:

In [ ]:
def classify_puzzle(prompt):
    """Classify a puzzle prompt into one of 6 categories based on keyword patterns."""
    prompt_lower = prompt.lower()

    if re.search(r"numeral system|base[- ]?\d|number.*convert|radix|secret number", prompt_lower):
        return "Number Base Conversion"
    elif re.search(r"gravit|gravity|falling|free.?fall|acceleration due to", prompt_lower):
        return "Gravitational Constant"
    elif re.search(
        r"transformation rule|equation.*transform|secret.*rule.*equation|rule.*applied.*equation",
        prompt_lower,
    ):
        return "Equation Transformation"
    elif re.search(
        r"encrypt|cipher|secret.*code.*letter|coded.*message|secret.*text", prompt_lower
    ):
        return "Text Encryption"
    elif re.search(r"bit.?manipul|binary|8.?bit|bitwise|bit.*transform", prompt_lower):
        return "Bit Manipulation"
    elif re.search(r"unit.?conver|measurement|becomes.*\d|secret.*conver.*measur", prompt_lower):
        return "Unit Conversion"
    else:
        return "Unknown"


# Apply classification
train_df["puzzle_type"] = train_df["prompt"].apply(classify_puzzle)

# Check results
type_counts = train_df["puzzle_type"].value_counts()
print("Puzzle Type Distribution:")
print("-" * 45)
for ptype, count in type_counts.items():
    pct = count / len(train_df) * 100
    print(f"  {ptype:<30s} {count:>4d}  ({pct:.1f}%)")

# Flag any unclassified rows
unknown = train_df[train_df["puzzle_type"] == "Unknown"]
if len(unknown) > 0:
    print(f"\nWARNING: {len(unknown)} rows could not be classified! Inspect them below:")
    for idx, row in unknown.head(3).iterrows():
        print(f"\n  Row {idx}: {row['prompt'][:200]}...")
else:
    print(f"\nAll {len(train_df)} rows classified successfully!")

In [ ]:
# Visualize the distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart
colors = [PUZZLE_COLORS.get(t, "#999") for t in type_counts.index]
bars = axes[0].barh(
    type_counts.index, type_counts.values, color=colors, edgecolor="white", height=0.6
)
axes[0].set_xlabel("Number of Puzzles", fontsize=12)
axes[0].set_title("Puzzle Count by Type", fontsize=14, fontweight="bold")
for bar, count in zip(bars, type_counts.values, strict=False):
    axes[0].text(
        bar.get_width() + 5,
        bar.get_y() + bar.get_height() / 2,
        str(count),
        va="center",
        fontsize=11,
        fontweight="bold",
    )
axes[0].invert_yaxis()

# Pie chart
axes[1].pie(
    type_counts.values,
    labels=type_counts.index,
    autopct="%1.1f%%",
    colors=colors,
    startangle=90,
    textprops={"fontsize": 10},
)
axes[1].set_title("Puzzle Type Distribution", fontsize=14, fontweight="bold")

plt.suptitle("The 6 Puzzle Types in the Training Set", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 3. Deep Dive: Each Puzzle Type

Let's examine each type in detail: what the prompts look like, how answers are formatted, and what reasoning skills are needed.

In [ ]:
def analyze_puzzle_type(df, puzzle_type):
    """Print a detailed analysis of one puzzle type."""
    subset = df[df["puzzle_type"] == puzzle_type]

    print(f"{'=' * 80}")
    print(f"  PUZZLE TYPE: {puzzle_type}")
    print(f"  Count: {len(subset)} puzzles")
    print(f"{'=' * 80}")

    # Prompt stats
    print("\nPrompt Length:")
    print(
        f"   Characters - min: {subset['prompt_len'].min()}, "
        f"median: {subset['prompt_len'].median():.0f}, max: {subset['prompt_len'].max()}"
    )
    print(f"   Est. tokens - median: {subset['est_tokens'].median():.0f}")

    # Answer analysis
    answers = subset["answer"].astype(str)
    numeric_answers = answers[answers.apply(lambda x: bool(re.match(r"^-?\d+\.?\d*$", x.strip())))]
    print("\nAnswer Format:")
    print(f"   Total answers: {len(answers)}")
    print(
        f"   Numeric answers: {len(numeric_answers)} ({len(numeric_answers)/len(answers)*100:.1f}%)"
    )
    print(f"   Non-numeric answers: {len(answers) - len(numeric_answers)}")

    # Sample answers
    print("\nSample Answers (first 10):")
    print(f"   {list(subset['answer'].head(10))}")

    # Sample prompt (truncated)
    print("\nSample Prompt (truncated to 600 chars):")
    print(f"   {subset['prompt'].iloc[0][:600]}...")
    print()


for ptype in type_counts.index:
    analyze_puzzle_type(train_df, ptype)

In [ ]:
# Compare prompt lengths across puzzle types
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Box plot of prompt lengths by type
order = type_counts.index.tolist()
palette = [PUZZLE_COLORS.get(t, "#999") for t in order]

sns.boxplot(
    data=train_df,
    y="puzzle_type",
    x="prompt_len",
    order=order,
    palette=palette,
    ax=axes[0],
    width=0.6,
)
axes[0].set_title("Prompt Length by Puzzle Type (chars)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Characters")
axes[0].set_ylabel("")

# Box plot of answer lengths by type
sns.boxplot(
    data=train_df,
    y="puzzle_type",
    x="answer_len",
    order=order,
    palette=palette,
    ax=axes[1],
    width=0.6,
)
axes[1].set_title("Answer Length by Puzzle Type (chars)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Characters")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

## 4. Answer Format Analysis

This is **critical**: the evaluation extracts the answer from `\boxed{}`. We need to understand exactly what format each puzzle type expects.

In [ ]:
def classify_answer_format(answer):
    """Determine the format of an answer string."""
    answer = str(answer).strip()
    if re.match(r"^[01]{8}$", answer):
        return "binary_8bit"
    elif re.match(r"^-?\d+$", answer):
        return "integer"
    elif re.match(r"^-?\d+\.\d+$", answer):
        return "decimal"
    elif re.match(r"^[a-zA-Z\s]+$", answer):
        return "text"
    elif re.match(r"^[0-9a-fA-F]+$", answer):
        return "hex_or_int"
    else:
        return "other"


train_df["answer_format"] = train_df["answer"].apply(classify_answer_format)

# Cross-tabulation: puzzle type vs answer format
ct = pd.crosstab(train_df["puzzle_type"], train_df["answer_format"])
print("Answer Format by Puzzle Type:")
print(ct.to_string())

# Heatmap visualization
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(ct, annot=True, fmt="d", cmap="YlOrRd", linewidths=0.5, ax=ax)
ax.set_title("Answer Format Distribution by Puzzle Type", fontsize=14, fontweight="bold")
ax.set_ylabel("Puzzle Type")
ax.set_xlabel("Answer Format")
plt.tight_layout()
plt.show()

## 5. Pattern Mining: Reverse-Engineering the Hidden Rules

The key insight: each puzzle gives **several input→output examples** and asks you to apply the same hidden rule to a new input. Let's extract these patterns to understand what the model needs to learn.

In [ ]:
# --- UNIT CONVERSION: Extract the hidden conversion factor ---
print("=" * 80)
print("  PATTERN MINING: Unit Conversion")
print("=" * 80)

unit_conv = train_df[train_df["puzzle_type"] == "Unit Conversion"]
if len(unit_conv) > 0:
    sample = unit_conv.iloc[0]
    prompt = sample["prompt"]

    # Extract input→output pairs (e.g., "14.89 m becomes 21.91")
    pairs = re.findall(r"(\d+\.?\d*)\s*(?:m|kg|s|cm|km|L)?\s*becomes\s*(\d+\.?\d*)", prompt)
    if pairs:
        factors = [float(b) / float(a) for a, b in pairs if float(a) != 0]
        avg_factor = np.mean(factors)
        std_factor = np.std(factors)
        print(f"\n  Extracted {len(pairs)} example pairs:")
        for a, b in pairs:
            print(f"    {a} → {b}  (factor: {float(b)/float(a):.6f})")
        print(f"\n  Average conversion factor: {avg_factor:.6f} (std: {std_factor:.6f})")
        print("  The model needs to: compute this ratio and apply it to the target value")

    # Extract the target
    target_match = re.search(r"convert.*?:\s*(\d+\.?\d*)", prompt, re.I)
    if target_match:
        target = float(target_match.group(1))
        predicted = target * avg_factor
        print(f"\n  Target: {target}")
        print(f"  Predicted answer: {predicted:.2f}")
        print(f"  Actual answer:    {sample['answer']}")

print(f"\n\n{'=' * 80}")
print("  PATTERN MINING: Number Base Conversion")
print("=" * 80)

num_base = train_df[train_df["puzzle_type"] == "Number Base Conversion"]
if len(num_base) > 0:
    sample = num_base.iloc[0]
    print("\n  Sample prompt (first 500 chars):")
    print(f"  {sample['prompt'][:500]}")
    print(f"\n  Answer: {sample['answer']}")
    print("\n  The model needs to: identify the target base, convert the number")

print(f"\n\n{'=' * 80}")
print("  PATTERN MINING: Bit Manipulation")
print("=" * 80)

bit_manip = train_df[train_df["puzzle_type"] == "Bit Manipulation"]
if len(bit_manip) > 0:
    sample = bit_manip.iloc[0]
    print("\n  Sample prompt (first 500 chars):")
    print(f"  {sample['prompt'][:500]}")
    print(f"\n  Answer: {sample['answer']}")

    # Try to extract binary pairs
    bin_pairs = re.findall(
        r"([01]{8})\s*(?:becomes|maps to|transforms to|→|->)\s*([01]{8})", sample["prompt"]
    )
    if bin_pairs:
        print(f"\n  Extracted {len(bin_pairs)} binary transformations:")
        for a, b in bin_pairs[:5]:
            print(f"    {a} → {b}  (decimal: {int(a,2)} → {int(b,2)})")
        print("\n  The model needs to: discover the bitwise operation rule")

## 6. Difficulty Estimation

Based on community findings (zero-shot testing) and the nature of each puzzle type, let's estimate difficulty and prioritize training effort.

In [ ]:
# Difficulty estimation based on community zero-shot findings + analysis
difficulty_data = pd.DataFrame(
    {
        "Puzzle Type": [
            "Number Base Conversion",
            "Unit Conversion",
            "Gravitational Constant",
            "Equation Transformation",
            "Text Encryption",
            "Bit Manipulation",
        ],
        "Zero-Shot Accuracy (est.)": ["~100%", "~33%", "~33%", "~10%", "~5%", "~5%"],
        "Reasoning Required": [
            "Detect base, convert",
            "Compute ratio, multiply",
            "Identify g, apply formula",
            "Parse rules, apply algebra",
            "Decode cipher, apply to text",
            "Find bitwise op, apply",
        ],
        "Answer Type": [
            "Integer/Decimal",
            "Decimal",
            "Decimal",
            "Expression/Number",
            "Text",
            "Binary 8-bit",
        ],
        "Difficulty": ["Easy", "Medium", "Medium", "Hard", "Hard", "Hard"],
        "Training Priority": ["Low", "Medium", "Medium", "HIGH", "HIGH", "HIGH"],
    }
)

# Count in training set
type_count_map = train_df["puzzle_type"].value_counts().to_dict()
difficulty_data["Train Count"] = (
    difficulty_data["Puzzle Type"].map(type_count_map).fillna(0).astype(int)
)

print("Difficulty Estimation & Training Priority")
print("=" * 110)
print(difficulty_data.to_string(index=False))

# Visualize difficulty
fig, ax = plt.subplots(figsize=(12, 5))
difficulty_order = ["Easy", "Medium", "Hard"]
diff_colors = {"Easy": "#4ECDC4", "Medium": "#FFEAA7", "Hard": "#FF6B6B"}

for i, (_, row) in enumerate(difficulty_data.iterrows()):
    color = diff_colors[row["Difficulty"]]
    ax.barh(i, row["Train Count"], color=color, edgecolor="white", height=0.6)
    ax.text(
        row["Train Count"] + 2,
        i,
        f"  {row['Difficulty']} | Priority: {row['Training Priority']}",
        va="center",
        fontsize=10,
    )

ax.set_yticks(range(len(difficulty_data)))
ax.set_yticklabels(difficulty_data["Puzzle Type"], fontsize=11)
ax.set_xlabel("Number of Training Examples", fontsize=12)
ax.set_title("Training Priority by Puzzle Type Difficulty", fontsize=14, fontweight="bold")
ax.invert_yaxis()

# Legend
from matplotlib.patches import Patch

legend_elements = [Patch(facecolor=c, label=d) for d, c in diff_colors.items()]
ax.legend(handles=legend_elements, loc="lower right", fontsize=10)
plt.tight_layout()
plt.show()

## 7. SFT Training Data Preparation

Now let's prepare the training data in **chat format** for Supervised Fine-Tuning. This is the first step in the recommended two-stage pipeline (SFT → GRPO).

The key elements:
- **System prompt**: Instruct the model to reason step-by-step and output `\boxed{answer}`
- **User message**: The puzzle prompt
- **Assistant message**: A chain-of-thought reasoning trace + the boxed answer

This JSONL can be directly used with Hugging Face TRL, Unsloth, Axolotl, or NeMo for LoRA fine-tuning.

In [ ]:
SYSTEM_PROMPT = (
    "You are a precise reasoning assistant. You will be given a puzzle that describes "
    "a hidden transformation rule with several input-output examples. Your task is to:\n"
    "1. Carefully analyze the given examples to discover the hidden rule.\n"
    "2. State the rule you discovered.\n"
    "3. Apply the rule to the given input.\n"
    "4. Provide your final answer inside \\boxed{}.\n\n"
    "Think step by step. Be concise but thorough."
)


def create_reasoning_template(puzzle_type, answer):
    """Generate a minimal chain-of-thought template based on puzzle type."""
    answer_str = str(answer)

    templates = {
        "Number Base Conversion": (
            "Let me analyze the examples to find the pattern.\n\n"
            "Looking at the input-output pairs, I need to identify what numeral system "
            "conversion is being applied.\n\n"
            "After examining the examples, I can see the conversion rule.\n\n"
            "Applying this rule to the given input:\n\n"
            f"\\boxed{{{answer_str}}}"
        ),
        "Gravitational Constant": (
            "Let me analyze the examples to find the hidden gravitational constant.\n\n"
            "From the input-output pairs, I can compute the modified value of g "
            "by comparing expected vs actual results.\n\n"
            "Applying the modified gravitational constant to the given problem:\n\n"
            f"\\boxed{{{answer_str}}}"
        ),
        "Equation Transformation": (
            "Let me analyze the transformation rules applied to these equations.\n\n"
            "Looking at each example, I need to identify what algebraic "
            "transformations are being applied.\n\n"
            "The transformation rules I've identified are applied to the given equation:\n\n"
            f"\\boxed{{{answer_str}}}"
        ),
        "Text Encryption": (
            "Let me analyze the encryption pattern in the examples.\n\n"
            "By comparing the input text with the encrypted output, I can identify "
            "the cipher or substitution rule being used.\n\n"
            "Applying the encryption rule to the given text:\n\n"
            f"\\boxed{{{answer_str}}}"
        ),
        "Bit Manipulation": (
            "Let me analyze the bit transformation pattern.\n\n"
            "Converting the binary examples to decimal and examining the relationship "
            "between inputs and outputs to identify the bitwise operation.\n\n"
            "Applying the bit manipulation rule to the given binary number:\n\n"
            f"\\boxed{{{answer_str}}}"
        ),
        "Unit Conversion": (
            "Let me analyze the secret unit conversion.\n\n"
            "Computing the ratio between output and input for each example pair "
            "to find the conversion factor.\n\n"
            "Applying the conversion factor to the given measurement:\n\n"
            f"\\boxed{{{answer_str}}}"
        ),
    }
    return templates.get(puzzle_type, f"After analyzing the pattern: \\boxed{{{answer_str}}}")


# Build the chat-formatted dataset
sft_data = []
for _, row in train_df.iterrows():
    assistant_response = create_reasoning_template(row["puzzle_type"], row["answer"])

    conversation = {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": row["prompt"]},
            {"role": "assistant", "content": assistant_response},
        ],
        "puzzle_type": row["puzzle_type"],
        "id": row["id"],
    }
    sft_data.append(conversation)

# Save as JSONL
output_path = "train_sft.jsonl"
with open(output_path, "w") as f:
    for item in sft_data:
        f.write(json.dumps(item) + "\n")

print(f"SFT training data saved to: {output_path}")
print(f"Total examples: {len(sft_data)}")
print("\n--- Sample entry ---")
sample = sft_data[0]
print(f"System: {sample['messages'][0]['content'][:100]}...")
print(f"User:   {sample['messages'][1]['content'][:100]}...")
print(f"Asst:   {sample['messages'][2]['content'][:200]}...")

In [ ]:
# Also create a category-balanced version (oversampling hard categories)
# This is important since the model already handles easy puzzles well
hard_types = ["Bit Manipulation", "Text Encryption", "Equation Transformation"]
medium_types = ["Gravitational Constant", "Unit Conversion"]

hard_df = train_df[train_df["puzzle_type"].isin(hard_types)]
medium_df = train_df[train_df["puzzle_type"].isin(medium_types)]
easy_df = train_df[~train_df["puzzle_type"].isin(hard_types + medium_types)]

# Oversample: hard x3, medium x2, easy x1
balanced_df = (
    pd.concat(
        [
            hard_df.sample(n=len(hard_df) * 3, replace=True, random_state=42),
            medium_df.sample(n=len(medium_df) * 2, replace=True, random_state=42),
            easy_df,
        ]
    )
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

print(f"Original training set: {len(train_df)} examples")
print(f"Balanced training set: {len(balanced_df)} examples")
print("\nBalanced distribution:")
print(balanced_df["puzzle_type"].value_counts().to_string())

# Save balanced version
balanced_sft = []
for _, row in balanced_df.iterrows():
    assistant_response = create_reasoning_template(row["puzzle_type"], row["answer"])
    conversation = {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": row["prompt"]},
            {"role": "assistant", "content": assistant_response},
        ],
        "puzzle_type": row["puzzle_type"],
        "id": row["id"],
    }
    balanced_sft.append(conversation)

with open("train_sft_balanced.jsonl", "w") as f:
    for item in balanced_sft:
        f.write(json.dumps(item) + "\n")

print("\nBalanced SFT data saved to: train_sft_balanced.jsonl")

## 8. Test Set: Apply Classifier & Verify Consistency

In [ ]:
# Classify test set puzzles too
test_df["puzzle_type"] = test_df["prompt"].apply(classify_puzzle)

print("Test Set Puzzle Type Distribution:")
print("-" * 45)
test_counts = test_df["puzzle_type"].value_counts()
for ptype, count in test_counts.items():
    pct = count / len(test_df) * 100
    print(f"  {ptype:<30s} {count:>4d}  ({pct:.1f}%)")

# Compare distributions
print("\n\nTrain vs Test Distribution Comparison:")
print("-" * 60)
comparison = pd.DataFrame(
    {
        "Train": train_df["puzzle_type"].value_counts(normalize=True).round(3),
        "Test (sample)": test_df["puzzle_type"].value_counts(normalize=True).round(3),
    }
)
comparison["Difference"] = (comparison["Train"] - comparison["Test (sample)"]).abs()
print(comparison.to_string())
print(
    "\nNote: The sample test set is small. The actual test set will be much larger with ~similar distribution."
)

## 9. Summary Dashboard

In [ ]:
# Create a comprehensive summary visualization
fig = plt.figure(figsize=(18, 10))

# 1. Puzzle type distribution (top left)
ax1 = fig.add_subplot(2, 2, 1)
colors = [PUZZLE_COLORS.get(t, "#999") for t in type_counts.index]
bars = ax1.barh(type_counts.index, type_counts.values, color=colors, edgecolor="white")
ax1.set_title("Training Set: Puzzle Counts", fontweight="bold")
ax1.invert_yaxis()
for bar, count in zip(bars, type_counts.values, strict=False):
    ax1.text(
        bar.get_width() + 1, bar.get_y() + bar.get_height() / 2, str(count), va="center", fontsize=9
    )

# 2. Prompt length distributions (top right)
ax2 = fig.add_subplot(2, 2, 2)
for ptype in type_counts.index:
    subset = train_df[train_df["puzzle_type"] == ptype]["prompt_len"]
    ax2.hist(subset, bins=20, alpha=0.5, label=ptype, color=PUZZLE_COLORS.get(ptype, "#999"))
ax2.set_title("Prompt Length Distribution by Type", fontweight="bold")
ax2.set_xlabel("Characters")
ax2.legend(fontsize=7, loc="upper right")

# 3. Answer length by type (bottom left)
ax3 = fig.add_subplot(2, 2, 3)
summary_data = (
    train_df.groupby("puzzle_type")
    .agg(
        avg_prompt_len=("prompt_len", "mean"),
        avg_answer_len=("answer_len", "mean"),
        count=("id", "count"),
    )
    .reindex(type_counts.index)
)
x = range(len(summary_data))
ax3.bar(x, summary_data["avg_answer_len"], color=colors, edgecolor="white")
ax3.set_xticks(x)
ax3.set_xticklabels(summary_data.index, rotation=30, ha="right", fontsize=8)
ax3.set_title("Average Answer Length by Type", fontweight="bold")
ax3.set_ylabel("Characters")

# 4. Key metrics table (bottom right)
ax4 = fig.add_subplot(2, 2, 4)
ax4.axis("off")
table_data = []
for ptype in type_counts.index:
    subset = train_df[train_df["puzzle_type"] == ptype]
    table_data.append(
        [
            ptype[:20],
            len(subset),
            f"{subset['prompt_len'].median():.0f}",
            f"{subset['answer_len'].median():.0f}",
            difficulty_data[difficulty_data["Puzzle Type"] == ptype]["Difficulty"].values[0],
        ]
    )

table = ax4.table(
    cellText=table_data,
    colLabels=["Puzzle Type", "Count", "Med. Prompt", "Med. Answer", "Difficulty"],
    loc="center",
    cellLoc="center",
)
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.2, 1.5)
ax4.set_title("Summary Statistics", fontweight="bold", pad=20)

plt.suptitle(
    "NVIDIA Nemotron Reasoning Challenge - Data Overview", fontsize=16, fontweight="bold", y=1.01
)
plt.tight_layout()
plt.show()

## 10. Key Takeaways & Strategy Recommendations

### What We Learned
1. **6 puzzle types**: all present in both train and test with similar distributions
2. **Easy wins**: Number Base Conversion (~100% zero-shot) - don't waste training budget here
3. **Biggest gaps**: Bit Manipulation, Text Encryption, Equation Transformation (~5-10% zero-shot) - **this is where training will have the most impact**
4. **Answers are short**: integers, decimals, 8-bit binary, or short text strings

### Recommended Training Pipeline

| Stage | What | Why |
|-------|------|-----|
| **1. SFT** | Fine-tune LoRA on the chat-formatted training data | Teach the model to consistently use `\boxed{}` format and structured reasoning |
| **2. Data Augmentation** | Generate synthetic puzzles (especially for hard types) using a teacher model | More training signal for the weakest categories |
| **3. GRPO/RL** | Use reward-based optimization (correct answer → positive reward) | Refine reasoning quality beyond imitation |

### Important Constraints to Remember
- **Max LoRA rank: 32** - keep it efficient
- **Max tokens: 7,680** - keep reasoning traces compact, not verbose
- **Max model length: 8,192** - prompt + response must fit in this window
- **Temperature: 0.0** - deterministic inference, no sampling tricks
- **Only submit a LoRA adapter** - no prompt engineering at inference time

### Output Files
- `train_sft.jsonl` - Chat-formatted training data (original distribution)> 
- `train_sft_balanced.jsonl` - Oversampled hard categories (recommended for training)

---

**If this notebook helped you, please upvote! Good luck to everyone!**